<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

This is a ranking / scoring problem. The decision is not simply “yes or no,” but “which pages should a content reviewer tackle first?” A model can assign each page a priority score and the team can work through the top-ranked pages in order. That makes this a ranked queue problem rather than a single-threshold classification task.

In [ ]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Sample row:")
print(df.head(1).to_string(index=False))


Rows: 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Sample row:
          content_id         client_id  search_volume  competition competition_level  cpc    content_type   main_intent  word_count  char_count provider_used       model_used  impressions_90d  cli

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

The target I would predict is a page-level priority score for likely refresh need. In this starter dataset, the label is an observed outcome from the later trend signal: whether the page later shows a declining trend, captured in the column `trend_direction`. That is a real observed outcome, not a hand-written rule, so it is appropriate for a supervised learning setup. The model would learn patterns that are associated with later decline and use that to rank pages for review.

In [ ]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
label = (df["trend_direction"] == "down").astype(int)
print("Observed decline label share:", label.mean())
print("Observed decline count:", label.sum())


Observed decline label share: 0.5420666666666667
Observed decline count: 16262


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

A defensible metric here is precision@K, because the team will act on a short ranked list. If the model places the most promising refresh candidates near the top of the queue, that is useful even if it does not perfectly classify every page. For this assignment, a strong result would be a precision@50 that is clearly above the simple baseline and improves the ranking quality of the top 50 pages.

In [8]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
label = (df["trend_direction"] == "down").astype(int)
print("Baseline rate for decline labels:", round(label.mean(), 3))
print("If we scored 50 pages at random, expected true positives would be about:", round(label.mean() * 50, 1))
summary = pd.DataFrame({
    "Metric": [
        "Total pages",
        "Pages declining",
        "Share declining",
        "Pages with impressions_90d >= 500"
    ],
    "Value": [
        len(df),
        (df['trend_direction'] == 'down').sum(),
        "{:.1%}".format((df['trend_direction'] == 'down').mean()),
        (df['impressions_90d'] >= 500).sum()
    ]
})

summary


Baseline rate for decline labels: 0.542
If we scored 50 pages at random, expected true positives would be about: 27.1


,Metric,Value
0,Total pages,30000
1,Pages declining,16262
2,Share declining,54.2%
3,Pages with impressions_90d >= 500,16726


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

Each row is one page. The dataset contains page-level features such as impressions, click-through rate, position, freshness, and the observed later trend direction. The unit of analysis is therefore the page, and the model will score one page at a time for refresh priority.

In [ ]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.head(3).to_string(index=False))

          content_id         client_id  search_volume  competition competition_level  cpc    content_type   main_intent  word_count  char_count provider_used             model_used  impressions_90d  clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  scroll_events_90d  days_with_impressions  days_with_sessions  impressions_last_30d  clicks_last_30d  sessions_last_30d  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  content_age_days age_tier  age_tier_order  days_since_last_update freshness_tier word_count_tier char_count_tier  ctr  avg_position  engagement_rate  scroll_rate  ai_traffic_pct impression_tier position_tier trend_direction  trend_pct
content_304f48230142 client_f369cb89fc           10.0         0.67              HIGH 2.05 keyword article transactional      3221.0     20457.0           NaN       gemini-2.5-flash             3803          29             22            17         16                     1                0         

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A fixed rule is too brittle because the signals that matter are mixed and non-linear. A page with strong impressions but weak recent engagement may need a different response than a page with lower impressions but a sharp recent drop. The relationship between signals and refresh need is not a simple if-then rule, so a model can learn a more nuanced ranking function from the observed patterns.

In [ ]:
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Decline share by visibility band:")
print(pd.qcut(df["impressions_90d"], q=4, duplicates="drop").value_counts().to_string())
df["impression_band"] = pd.qcut(df["impressions_90d"], q=4, duplicates="drop")
print(df.groupby("impression_band", observed=True)["trend_direction"].apply(lambda s: (s == "down").mean()))

Decline share by visibility band:
impressions_90d
(0.999, 81.0]          7503
(3615.25, 517715.0]    7500
(81.0, 731.0]          7499
(731.0, 3615.25]       7498
impression_band
(0.999, 81.0]          0.376116
(81.0, 731.0]          0.604614
(731.0, 3615.25]       0.625634
(3615.25, 517715.0]    0.562000
Name: trend_direction, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.